<a href="https://colab.research.google.com/github/hema-chodisetti/AI-Support-Ticket-Intelligence/blob/main/AI_STIP_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate sentence-transformers sentencepiece torch

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    pipeline
)

from datasets import load_dataset

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

In [ ]:
category_model_path = "/content/drive/MyDrive/AI_STIP/models/category_model"
priority_model_path = "/content/drive/MyDrive/AI_STIP/models/priority_model"

In [ ]:
print("Category model exists:", os.path.exists(category_model_path))
print("Priority model exists:", os.path.exists(priority_model_path))

Category model exists: True
Priority model exists: True


In [ ]:
category_tokenizer = AutoTokenizer.from_pretrained(
    category_model_path,
    local_files_only=True
)

category_model = AutoModelForSequenceClassification.from_pretrained(
    category_model_path,
    local_files_only=True
)

category_classifier = pipeline(
    "text-classification",
    model=category_model,
    tokenizer=category_tokenizer
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
category_classifier(
    "I want to cancel my order"
)

[{'label': 'ORDER', 'score': 0.9994949102401733}]

In [ ]:
priority_tokenizer = AutoTokenizer.from_pretrained(
    priority_model_path,
    local_files_only=True
)

priority_model = AutoModelForSequenceClassification.from_pretrained(
    priority_model_path,
    local_files_only=True
)

priority_classifier = pipeline(
    "text-classification",
    model=priority_model,
    tokenizer=priority_tokenizer
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
priority_classifier(
    "Our production system is completely down and all customers are affected."
)

[{'label': 'high', 'score': 0.9432427883148193}]

In [ ]:
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
sentiment_classifier(
    "I am extremely frustrated because nobody has solved my issue."
)

[{'label': 'negative', 'score': 0.9315696954727173}]

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

summary_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

summary_tokenizer = AutoTokenizer.from_pretrained(
    summary_model_name
)

summary_model = AutoModelForCausalLM.from_pretrained(
    summary_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def summarize_ticket(ticket):

    messages = [
        {
            "role": "system",
            "content": (
                "You summarize customer support tickets for support agents. "
                "Return exactly one short sentence. "
                "Include the main problem and what the customer wants if stated. "
                "Remove unnecessary emotion and repetition. "
                "Never invent information or claim something was resolved. "
                "Do not include labels such as Summary, Ticket, Problem, or Request."
            )
        },
        {
            "role": "user",
            "content": (
                "Example:\n"
                "Ticket: I was billed twice and I am angry. "
                "Please refund the extra charge.\n"
                "Answer: Customer was charged twice and requests a refund "
                "for the duplicate charge.\n\n"
                f"Ticket: {ticket}\n"
                "Answer:"
            )
        }
    ]

    text = summary_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = summary_tokenizer(
        text,
        return_tensors="pt"
    ).to(summary_model.device)

    with torch.no_grad():
        outputs = summary_model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            repetition_penalty=1.1
        )

    # IMPORTANT:
    # Decode ONLY newly generated tokens,
    # not the original prompt.
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    summary = summary_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return summary

In [ ]:
test_ticket = """
I was charged twice for the same order and I am extremely frustrated.
I need the duplicate payment refunded immediately.
"""

print(summarize_ticket(test_ticket))

Customer was charged twice for the same order and requests immediate refund of both charges.


In [ ]:
dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

In [ ]:
tickets_data = dataset["train"]

old_tickets = tickets_data["instruction"]

print("Historical tickets:", len(old_tickets))

Historical tickets: 26872


In [ ]:
from collections import Counter

category_counts = Counter(tickets_data["category"])

print(category_counts)

Counter({'ACCOUNT': 5986, 'ORDER': 3988, 'REFUND': 2992, 'INVOICE': 1999, 'CONTACT': 1999, 'PAYMENT': 1998, 'FEEDBACK': 1997, 'DELIVERY': 1994, 'SHIPPING': 1970, 'SUBSCRIPTION': 999, 'CANCEL': 950})


In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
old_tickets = tickets_data["instruction"]

old_embeddings = embedding_model.encode(
    old_tickets,
    convert_to_tensor=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/840 [00:00<?, ?it/s]

In [ ]:
def find_similar_tickets(ticket, category, top_k=5):

    new_embedding = embedding_model.encode(
        ticket,
        convert_to_tensor=True
    )

    candidate_indices = [
        i
        for i in range(len(tickets_data))
        if tickets_data[i]["category"].upper() == category.upper()
    ]

    print(
        f"Searching {len(candidate_indices)} historical "
        f"tickets in category: {category}"
    )

    if len(candidate_indices) == 0:
        return []

    candidate_embeddings = old_embeddings[candidate_indices]

    similarities = cos_sim(
        new_embedding,
        candidate_embeddings
    )[0]

    top_results = similarities.argsort(
        descending=True
    )[:top_k]

    matches = []

    for local_index in top_results:

        local_index = local_index.item()
        original_index = candidate_indices[local_index]

        matches.append({
            "ticket": tickets_data[original_index]["instruction"],
            "category": tickets_data[original_index]["category"],
            "intent": tickets_data[original_index]["intent"],
            "previous_response": tickets_data[original_index]["response"],
            "similarity": round(
                similarities[local_index].item(),
                4
            )
        })

    return matches

In [ ]:
def find_similar_tickets(
    ticket,
    category,
    top_k=5
):

    new_embedding = embedding_model.encode(
        ticket,
        convert_to_tensor=True
    )

    # Ensure that candidate_indices do not exceed the actual size of old_embeddings
    max_embedding_index = len(old_embeddings)

    candidate_indices = [
        i
        for i in range(len(tickets_data))
        if tickets_data[i]["category"].upper() == category.upper() and i < max_embedding_index
    ]

    print(
        f"Searching {len(candidate_indices)} historical "
        f"tickets in category: {category} (limited by available embeddings: {max_embedding_index})"
    )

    if len(candidate_indices) == 0:
        return []

    candidate_embeddings = old_embeddings[candidate_indices]

    similarities = cos_sim(
        new_embedding,
        candidate_embeddings
    )[0]

    top_results = similarities.argsort(
        descending=True
    )[:top_k]

    matches = []

    for local_index in top_results:

        local_index = local_index.item()
        original_index = candidate_indices[local_index]

        matches.append({
            "ticket": tickets_data[original_index]["instruction"],
            "category": tickets_data[original_index]["category"],
            "intent": tickets_data[original_index]["intent"],
            "previous_response": tickets_data[original_index]["response"],
            "similarity": round(
                similarities[local_index].item(),
                4
            )
        })

    return matches

matches = find_similar_tickets(
    "I was charged twice for my purchase and want my money back.",
    category="REFUND"
)

for match in matches:
    print(match["similarity"], "-", match["ticket"])

Searching 2992 historical tickets in category: REFUND (limited by available embeddings: 26872)
0.6904 - I have to receive my money back
0.6836 - help to get my money back
0.683 - I would like to obtain my money back
0.6809 - want help to get my money back
0.6722 - were do i obtain my money back


In [ ]:
def generate_suggested_response(
    ticket,
    category,
    priority,
    sentiment,
    summary,
    similar_tickets
):

    reference_text = ""

    # Only use top 2 examples
    for i, match in enumerate(similar_tickets[:2], start=1):

        reference_text += f"""
Example {i}

Similar customer issue:
{match["ticket"]}

Historical response:
{match["previous_response"][:500]}
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a professional customer support assistant. "
                "Write a short response that a human support agent can review before sending. "
                "Acknowledge the customer's actual problem and explain a safe next step. "
                "Use historical responses only for general context. "
                "Do not copy procedures from historical responses unless the current information confirms them. "
                "Do not invent website buttons, account menus, company policies, refund status, "
                "fees, phone numbers, dates, guarantees, or actions that already happened. "
                "Use one short paragraph of about 2 to 4 sentences."
            )
        },
        {
            "role": "user",
            "content": f"""
Current ticket:
{ticket}

Category:
{category}

Priority:
{priority}

Sentiment:
{sentiment}

Summary:
{summary}

Historical context:
{reference_text}

Write only the suggested response to the customer.
"""
        }
    ]

    text = summary_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = summary_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2500
    ).to(summary_model.device)

    with torch.no_grad():

        outputs = summary_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            repetition_penalty=1.1
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    response = summary_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
def analyze_ticket(ticket):

    # Phase 1 - Category
    category_result = category_classifier(ticket)[0]

    # Phase 2 - Priority
    priority_result = priority_classifier(ticket)[0]

    # Phase 3 - Sentiment
    sentiment_result = sentiment_classifier(ticket)[0]

    # Phase 4 - Summary
    summary = summarize_ticket(ticket)

    # Phase 5 - Similar Tickets
    similar_tickets = find_similar_tickets(
    ticket,
    category=category_result["label"],
    top_k=5
   )

    # Phase 6 - Suggested Response
    suggested_response = generate_suggested_response(
        ticket=ticket,
        category=category_result["label"],
        priority=priority_result["label"],
        sentiment=sentiment_result["label"],
        summary=summary,
        similar_tickets=similar_tickets
    )

    return {
        "ticket": ticket,

        "category": category_result["label"],
        "category_confidence": round(
            category_result["score"],
            4
        ),

        "priority": priority_result["label"],
        "priority_confidence": round(
            priority_result["score"],
            4
        ),

        "sentiment": sentiment_result["label"],
        "sentiment_confidence": round(
            sentiment_result["score"],
            4
        ),

        "summary": summary,

        "similar_tickets": similar_tickets,

        "suggested_response": suggested_response
    }

In [ ]:
def analyze_for_ui(ticket):

    if not ticket.strip():
        return (
            "",
            "",
            "",
            "",
            "",
            "Please enter a customer support ticket."
        )

    result = analyze_and_save(ticket)

    category = (
        f'{result["category"]} '
        f'({result["category_confidence"]:.2%} confidence)'
    )

    priority = (
        f'{result["priority"]} '
        f'({result["priority_confidence"]:.2%} confidence)'
    )

    sentiment = (
        f'{result["sentiment"]} '
        f'({result["sentiment_confidence"]:.2%} confidence)'
    )

    summary = result["summary"]

    suggested_response = result["suggested_response"]

    similar_text = ""

    for i, match in enumerate(
        result["similar_tickets"],
        start=1
    ):

        similar_text += f"""
### Match {i}

**Similarity:** {match["similarity"]:.2%}

**Ticket:**
{match["ticket"]}

**Category:** {match["category"]}

**Intent:** {match["intent"]}

**Previous Response:**
{match["previous_response"]}

---
"""

    return (
        category,
        priority,
        sentiment,
        summary,
        suggested_response,
        similar_text
    )

In [ ]:
ticket = input(
    "Enter your customer support ticket: "
)

result = analyze_ticket(ticket)

Enter your customer support ticket: I was charged twice for the same order and need the duplicate payment refunded.
Searching 2992 historical tickets in category: REFUND (limited by available embeddings: 26872)


In [ ]:
print("\n======================================")
print("     AI SUPPORT TICKET ANALYSIS")
print("======================================")

print("\nOriginal Ticket:")
print(result["ticket"])

print("\nCategory:")
print(
    result["category"],
    "| Confidence:",
    result["category_confidence"]
)

print("\nPriority:")
print(
    result["priority"],
    "| Confidence:",
    result["priority_confidence"]
)

print("\nSentiment:")
print(
    result["sentiment"],
    "| Confidence:",
    result["sentiment_confidence"]
)

print("\nSummary:")
print(result["summary"])
print("\n======================================")
print("        SUGGESTED RESPONSE")
print("======================================")

print(result["suggested_response"])

print("\n======================================")
print("        TOP 5 SIMILAR TICKETS")
print("======================================")

for i, match in enumerate(
    result["similar_tickets"],
    start=1
):

    print(f"\n----- Match {i} -----")

    print("Similarity:", match["similarity"])

    print("\nTicket:")
    print(match["ticket"])

    print("\nCategory:")
    print(match["category"])

    print("\nIntent:")
    print(match["intent"])

    print("\nPrevious Response:")
    print(match["previous_response"])




     AI SUPPORT TICKET ANALYSIS

Original Ticket:
I was charged twice for the same order and need the duplicate payment refunded.

Category:
REFUND | Confidence: 0.9372

Priority:
medium | Confidence: 0.5088

Sentiment:
negative | Confidence: 0.8222

Summary:
Customer was charged twice for the same order and requests a refund for the duplicate payment.

        SUGGESTED RESPONSE
Thank you for reaching out regarding your charge issues. We apologize for any inconvenience caused by the duplication of payments. Please provide the order number associated with the duplicate transaction so we can proceed with the refund. Additionally, let us know if there is any additional information you believe might be helpful in resolving this matter.

        TOP 5 SIMILAR TICKETS

----- Match 1 -----
Similarity: 0.6152

Ticket:
I paid {{Refund Amount}} dollars for this order, I need a compensation

Category:
REFUND

Intent:
get_refund

Previous Response:
I see your frustration and desire for compensat

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

In [ ]:
with gr.Blocks(
    title="AI Support Ticket Intelligence Platform"
) as demo:

    gr.Markdown(
        """
        # 🤖 AI Support Ticket Intelligence Platform

        Analyze customer support tickets using AI.
        """
    )

    with gr.Tabs():

        # ============================
        # TAB 1 - ANALYZE TICKET
        # ============================

        with gr.Tab("Analyze Ticket"):

            ticket_input = gr.Textbox(
                label="Customer Support Ticket",
                placeholder="Example: I was charged twice for my order...",
                lines=5
            )

            analyze_button = gr.Button(
                "Analyze Ticket",
                variant="primary"
            )

            gr.Markdown("## Analysis")

            with gr.Row():

                category_output = gr.Textbox(
                    label="Category"
                )

                priority_output = gr.Textbox(
                    label="Priority"
                )

                sentiment_output = gr.Textbox(
                    label="Sentiment"
                )

            summary_output = gr.Textbox(
                label="Ticket Summary",
                lines=3
            )

            suggested_response_output = gr.Textbox(
                label="Suggested Response",
                lines=6
            )

            gr.Markdown("## Similar Historical Tickets")

            similar_output = gr.Markdown()

        # ============================
        # TAB 2 - TICKET HISTORY
        # ============================

        with gr.Tab("Ticket History"):

            gr.Markdown(
                "## Previous Support Tickets"
            )

            refresh_button = gr.Button(
                "Refresh History"
            )

            history_table = gr.Dataframe(
                headers=[
                    "ID",
                    "Ticket",
                    "Category",
                    "Priority",
                    "Sentiment",
                    "Summary",
                    "Created At"
                ],
                interactive=False
            )

    # Analyze ticket
    analyze_button.click(
        fn=analyze_for_ui,
        inputs=ticket_input,
        outputs=[
            category_output,
            priority_output,
            sentiment_output,
            summary_output,
            suggested_response_output,
            similar_output
        ]
    )

    # Refresh ticket history
    refresh_button.click(
        fn=get_ticket_history,
        inputs=[],
        outputs=history_table
    )

In [ ]:
demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c76ee0a027924a50c3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import sqlite3
import os

db_path = "/content/drive/MyDrive/AI_STIP/tickets.db"

os.makedirs(
    "/content/drive/MyDrive/AI_STIP",
    exist_ok=True
)

conn = sqlite3.connect(db_path)

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS tickets (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    original_ticket TEXT NOT NULL,

    category TEXT,
    category_confidence REAL,

    priority TEXT,
    priority_confidence REAL,

    sentiment TEXT,
    sentiment_confidence REAL,

    summary TEXT,

    suggested_response TEXT,

    similar_tickets TEXT,

    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

conn.commit()
conn.close()

print("Database ready:", db_path)

Database ready: /content/drive/MyDrive/AI_STIP/tickets.db


In [ ]:
import json

def save_analysis(result):

    conn = sqlite3.connect(db_path)

    cursor = conn.cursor()

    similar_tickets_json = json.dumps(
        result["similar_tickets"]
    )

    cursor.execute("""
        INSERT INTO tickets (
            original_ticket,
            category,
            category_confidence,
            priority,
            priority_confidence,
            sentiment,
            sentiment_confidence,
            summary,
            suggested_response,
            similar_tickets
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (

        result["ticket"],

        result["category"],
        result["category_confidence"],

        result["priority"],
        result["priority_confidence"],

        result["sentiment"],
        result["sentiment_confidence"],

        result["summary"],

        result["suggested_response"],

        similar_tickets_json
    ))

    ticket_id = cursor.lastrowid

    conn.commit()
    conn.close()

    return ticket_id

In [ ]:
def analyze_and_save(ticket):

    result = analyze_ticket(ticket)

    ticket_id = save_analysis(result)

    result["ticket_id"] = ticket_id

    return result

In [ ]:
ticket = "My package has not arrived even though it was supposed to arrive yesterday."

result = analyze_and_save(ticket)

print("Saved Ticket ID:", result["ticket_id"])
print("Category:", result["category"])
print("Summary:", result["summary"])

Searching 1994 historical tickets in category: DELIVERY (limited by available embeddings: 26872)
Saved Ticket ID: 3
Category: DELIVERY
Summary: Customer's package did not arrive despite being scheduled for delivery yesterday.


In [ ]:
def get_recent_tickets(limit=10):

    conn = sqlite3.connect(db_path)

    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            id,
            original_ticket,
            category,
            priority,
            sentiment,
            summary,
            created_at

        FROM tickets

        ORDER BY id DESC

        LIMIT ?
    """, (limit,))

    rows = cursor.fetchall()

    conn.close()

    return rows

In [ ]:
recent_tickets = get_recent_tickets()

for ticket in recent_tickets:

    print("\nID:", ticket[0])
    print("Ticket:", ticket[1])
    print("Category:", ticket[2])
    print("Priority:", ticket[3])
    print("Sentiment:", ticket[4])
    print("Summary:", ticket[5])
    print("Created:", ticket[6])


ID: 3
Ticket: My package has not arrived even though it was supposed to arrive yesterday.
Category: DELIVERY
Priority: medium
Sentiment: negative
Summary: Customer's package did not arrive despite being scheduled for delivery yesterday.
Created: 2026-09-11 14:37:51

ID: 2
Ticket: My package has not arrived even though it was supposed to arrive yesterday.
Category: DELIVERY
Priority: medium
Sentiment: negative
Summary: Customer's package did not arrive despite being scheduled for delivery yesterday.
Created: 2026-09-11 14:28:39

ID: 1
Ticket: My package has not arrived even though it was supposed to arrive yesterday.
Category: DELIVERY
Priority: medium
Sentiment: negative
Summary: Customer's package did not arrive despite being scheduled for delivery yesterday.
Created: 2026-09-11 02:16:01


In [ ]:
import pandas as pd
import sqlite3

def get_ticket_history(limit=100):

    conn = sqlite3.connect(db_path)

    query = """
    SELECT
        id,
        original_ticket,
        category,
        priority,
        sentiment,
        summary,
        created_at
    FROM tickets
    ORDER BY id DESC
    LIMIT ?
    """

    df = pd.read_sql_query(
        query,
        conn,
        params=(limit,)
    )

    conn.close()

    return df